# TSDAE (PubMed) → NLI: biomedical sentence embeddings

Runs the repo's pipeline end to end by calling its scripts; all logic lives in `tsdae_sbert_pubmed/`.
See `README.md` for what each step does.

1. *Runtime → Change runtime type → T4 GPU*.
2. Put this repo on your Google Drive (upload the folder, or `git clone` into Drive) and set `REPO_DIR` below.
3. Run all. Every step skips itself if its output already exists on Drive, so after a disconnect just run all again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/tsdae-sbert-pubmed'  # where this repo lives on Drive
QUICK = True  # True: 100k TSDAE sentences (quick test). False: 300k (full run).

RUN_NAME = 'quick' if QUICK else 'full'
OUT_DIR = f'/content/drive/MyDrive/tsdae-sbert-pubmed-outputs/{RUN_NAME}'  # data, models, results (persist on Drive)
RAW_DIR = '/content/pubmed_raw'  # raw XML on the VM's disk; re-downloaded if the VM resets
TSDAE_SENTENCES = 100_000 if QUICK else 300_000
ARGS = f'--out-dir "{OUT_DIR}" --raw-dir "{RAW_DIR}" --tsdae-sentences {TSDAE_SENTENCES}'

%cd {REPO_DIR}

In [ ]:
!pip -q install -r requirements.txt
!python -c "import sentence_transformers, transformers, torch; print(sentence_transformers.__version__, transformers.__version__, torch.__version__, torch.cuda.is_available())"

Optional: unit tests for the PubMed parser (a few seconds, no network).

In [ ]:
!python -m pytest -q tests/test_prepare_pubmed.py

## Data: download PubMed baseline files, parse, hold out 2,000 dev pairs, build TSDAE sentences

In [ ]:
!python -m tsdae_sbert_pubmed.prepare_pubmed {ARGS}

## Step 1: TSDAE on PubMed sentences (model B)

In [ ]:
!python -m tsdae_sbert_pubmed.train_tsdae {ARGS}

## Step 2: NLI training, control (C) and from TSDAE (D)

In [ ]:
!python -m tsdae_sbert_pubmed.train_nli {ARGS} --start none

In [ ]:
!python -m tsdae_sbert_pubmed.train_nli {ARGS} --start "{OUT_DIR}/models/tsdae"

## Evaluate A–D

In [ ]:
!python -m tsdae_sbert_pubmed.evaluate {ARGS}

import pandas as pd
pd.read_csv(f'{OUT_DIR}/results.csv', index_col=0)